# PDF Parser - IBM OCR 파이프라인 테스트

IBM LayoutPredictor를 사용하는 파이프라인 엔드포인트 테스트 노트북입니다.

## 지원 현황

| 엔드포인트 | IBM OCR 지원 | 설명 |
|---|---|---|
| `POST /ocr` | ✅ 지원 | `modelId=ibm` 파라미터로 선택 |
| `POST /process` | ✅ 지원 | `modelId=ibm` + `ibmConfidenceThreshold` 파라미터 |

## 노트북 구성

1. `/ocr` IBM 모드 테스트
2. `/process` IBM 모드 테스트
3. 결과 확인 (S3 다운로드 및 미리보기)

## 사전 준비

```bash
uv run uvicorn api:app --host 0.0.0.0 --port 3000
```

In [13]:
import requests
import json
import time
from pathlib import Path
from IPython.display import display, Markdown, Image as IPImage
import pandas as pd

API_BASE_URL = "http://localhost:3000"

# S3 경로 설정
INPUT_PDF = "s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0000J0_001.pdf"  # 처리할 PDF 경로
OUTPUT_BASE = "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/"   

print(f"API: {API_BASE_URL}")
print(f"입력: {INPUT_PDF}")
print(f"출력: {OUTPUT_BASE}")

API: http://localhost:3000
입력: s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0000J0_001.pdf
출력: s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/


In [14]:
# Health check
resp = requests.get(f"{API_BASE_URL}/health")
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

if resp.status_code == 200:
    print("\nAPI 서버 정상")
else:
    print("API 서버 접근 불가")

{
  "status": "healthy",
  "service": "document-parser-api",
  "version": "0.2.0",
  "supported_formats": {
    "pdf": [
      ".pdf"
    ],
    "office": [
      ".pptx",
      ".docx",
      ".xlsx",
      ".rtf",
      ".ods",
      ".odp",
      ".odt"
    ]
  }
}

API 서버 정상


---
## 1. `/ocr` IBM 모드 테스트

`modelId=ibm` 파라미터로 IBM LayoutPredictor를 사용합니다.

IBM 모드는 Docling 전체 파이프라인 대신 LayoutPredictor를 직접 사용하여
`Section-header`, `List-item`, `Key-Value Region` 등 더 세밀한 레이아웃 감지를 합니다.

In [15]:
# IBM OCR 요청
ocr_request = {
    "inputPath": INPUT_PDF,
    "outputPath": OUTPUT_BASE,
    "tableMode": "accurate",
    "generateBboxImages": True,
    "modelId": "ibm",                # IBM LayoutPredictor 사용
    "ibmConfidenceThreshold": 0.3,   # bbox 표시 최소 confidence
}

print("IBM OCR 요청:")
print(json.dumps(ocr_request, indent=2, ensure_ascii=False))

t0 = time.time()
resp = requests.post(f"{API_BASE_URL}/ocr", json=ocr_request, timeout=600)
elapsed = time.time() - t0

print(f"\n응답 코드: {resp.status_code} ({elapsed:.1f}s)")

if resp.status_code == 200:
    result = resp.json()
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print(resp.text)

IBM OCR 요청:
{
  "inputPath": "s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0000J0_001.pdf",
  "outputPath": "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/",
  "tableMode": "accurate",
  "generateBboxImages": true,
  "modelId": "ibm",
  "ibmConfidenceThreshold": 0.3
}

응답 코드: 200 (7.9s)
{
  "success": true,
  "message": "Successfully OCR processed R3_A0000J0_001.pdf",
  "textMarkdownUri": "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_text.md",
  "bboxImagesUris": [
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/bbox/R3_A0000J0_001_page004_bbox.jpg",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/bbox/R3_A0000J0_001_page001_bbox.jpg",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/bbox/R3_A0000J0_001_page002_bbox.jpg",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/bbox/R3_A0000J0_001_page003_bbo

In [16]:
# Docling vs IBM OCR 비교
results_compare = []

for layout in ["docling", "ibm"]:
    req = {
        "inputPath": INPUT_PDF,
        "outputPath": OUTPUT_BASE,
        "generateBboxImages": False,
        "modelId": layout,
    }
    t0 = time.time()
    resp = requests.post(f"{API_BASE_URL}/ocr", json=req, timeout=600)
    elapsed = time.time() - t0

    if resp.status_code == 200:
        r = resp.json()
        results_compare.append({
            "Layout Model": layout.upper(),
            "Pages": r["stats"]["pages"],
            "Figures": r["stats"]["figures"],
            "Tables": r["stats"]["tables"],
            "Time(s)": r["elapsedSeconds"],
        })
    else:
        results_compare.append({"Layout Model": layout.upper(), "Error": resp.status_code})

display(pd.DataFrame(results_compare))

,Layout Model,Pages,Figures,Tables,Time(s)
0,DOCLING,5,1,9,45.78
1,IBM,5,12,46,7.21


---
## 2. `/process` IBM 모드 테스트

`modelId=ibm`으로 전체 파이프라인(OCR + LLM 요약 + 마크다운 빌드)을 실행합니다.

### IBM 모드 주요 파라미터

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `modelId` | `"docling"` | `"ibm"` 으로 지정 |
| `llmModelId` | Haiku 4.5 | Bedrock LLM 모델 ID |
| `ibmConfidenceThreshold` | `0.3` | bbox 포함 최소 confidence (낮을수록 더 많은 박스 포함) |
| `noSummary` | `false` | `true`이면 LLM 요약 건너뜀 (빠른 OCR-only 처리) |

In [17]:
# IBM /process - noSummary=True (빠른 테스트용)
process_ibm_request = {
    "inputPath": INPUT_PDF,
    "outputPath": OUTPUT_BASE,
    "modelId": "ibm",
    "ibmConfidenceThreshold": 0.3,
    "noSummary": True,            # LLM 요약 없이 OCR + 마크다운만
    "tableMode": "accurate",
}

print("IBM /process 요청 (noSummary=True):")
print(json.dumps(process_ibm_request, indent=2, ensure_ascii=False))

t0 = time.time()
resp = requests.post(f"{API_BASE_URL}/process", json=process_ibm_request, timeout=600)
elapsed = time.time() - t0

print(f"\n응답 코드: {resp.status_code} ({elapsed:.1f}s)")

if resp.status_code == 200:
    result_no_summary = resp.json()
    print(json.dumps(result_no_summary, indent=2, ensure_ascii=False))
else:
    print(resp.text)

IBM /process 요청 (noSummary=True):
{
  "inputPath": "s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0000J0_001.pdf",
  "outputPath": "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/",
  "modelId": "ibm",
  "ibmConfidenceThreshold": 0.3,
  "noSummary": true,
  "tableMode": "accurate"
}

응답 코드: 200 (7.7s)
{
  "success": true,
  "message": "Successfully processed R3_A0000J0_001.pdf",
  "finalMarkdownUri": "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_final.md",
  "uploadedFiles": [
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_text.md",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_final.md",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/pictures/ibm_figure/R3_A0000J0_001_picture_1.png",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/table/img/R3_A0000J0_001_table_10.pn

In [18]:
# IBM /process - LLM 요약 포함 (전체 파이프라인)
process_ibm_full_request = {
    "inputPath": INPUT_PDF,
    "outputPath": OUTPUT_BASE,
    #"llmModelId": "ap-northeast-2.anthropic.claude-haiku-4-5-20251001-v1:0",
    "modelId": "ibm",
    "ibmConfidenceThreshold": 0.3,
    "noSummary": False,           # LLM 요약 포함
    "tableMode": "accurate",
}

print("IBM /process 요청 (전체 파이프라인):")
print(json.dumps(process_ibm_full_request, indent=2, ensure_ascii=False))

t0 = time.time()
resp = requests.post(f"{API_BASE_URL}/process", json=process_ibm_full_request, timeout=600)
elapsed = time.time() - t0

print(f"\n응답 코드: {resp.status_code} ({elapsed:.1f}s)")

if resp.status_code == 200:
    result_full = resp.json()
    print(json.dumps(result_full, indent=2, ensure_ascii=False))
else:
    print(resp.text)

IBM /process 요청 (전체 파이프라인):
{
  "inputPath": "s3://miraeasset-product-knowledge-graph/zeroin/r3/R3_A0000J0_001.pdf",
  "outputPath": "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/",
  "modelId": "ibm",
  "ibmConfidenceThreshold": 0.3,
  "noSummary": false,
  "tableMode": "accurate"
}

응답 코드: 200 (8.6s)
{
  "success": true,
  "message": "Successfully processed R3_A0000J0_001.pdf",
  "finalMarkdownUri": "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_final.md",
  "uploadedFiles": [
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_text.md",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_final.md",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/pictures/ibm_figure/R3_A0000J0_001_picture_1.png",
    "s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/table/img/R3_A0000J0_001_table_10.png",
 

In [19]:
# Docling vs IBM /process 비교 (noSummary=True로 빠르게)
import pandas as pd

compare_results = []

for layout in ["docling", "ibm"]:
    req = {
        "inputPath": INPUT_PDF,
        "outputPath": OUTPUT_BASE,
        "modelId": layout,
        "noSummary": True,
        "tableMode": "accurate",
    }
    t0 = time.time()
    resp = requests.post(f"{API_BASE_URL}/process", json=req, timeout=600)
    elapsed = time.time() - t0

    if resp.status_code == 200:
        r = resp.json()
        s = r.get("stats", {})
        compare_results.append({
            "Layout Model": layout.upper(),
            "Pages": s.get("pages", "-"),
            "Figures": s.get("figures", "-"),
            "Tables": s.get("tables", "-"),
            "Time(s)": r.get("elapsedSeconds", "-"),
            "Final MD URI": r.get("finalMarkdownUri", "-"),
        })
    else:
        compare_results.append({
            "Layout Model": layout.upper(),
            "Error": f"{resp.status_code}: {resp.text[:100]}",
        })

df = pd.DataFrame(compare_results)
display(df)

,Layout Model,Pages,Figures,Tables,Time(s),Final MD URI
0,DOCLING,5,1,9,45.55,s3://miraeasset-product-knowledge-graph/output...
1,IBM,5,1,11,7.77,s3://miraeasset-product-knowledge-graph/output...


---
## 3. 결과 확인

S3에서 최종 마크다운을 다운로드하여 내용을 미리봅니다.

In [20]:
from pdf_parser.s3_handler import S3Handler
import tempfile
from pathlib import Path

s3 = S3Handler()

# 위 /process IBM 전체 파이프라인 결과 URI 사용
# result_full 변수가 없으면 직접 URI를 지정하세요
try:
    final_md_uri = result_full["finalMarkdownUri"]
except NameError:
    pdf_stem = Path(INPUT_PDF).stem
    final_md_uri = OUTPUT_BASE.rstrip("/") + f"/{pdf_stem}/{pdf_stem}_final.md"

print(f"읽는 파일: {final_md_uri}")
content = s3.read_markdown(final_md_uri)

print(f"전체 길이: {len(content):,} chars")
print("\n--- 미리보기 (첫 2000자) ---")†
print(content[:2000])

읽는 파일: s3://miraeasset-product-knowledge-graph/output_md/TEST_IBM/R3_A0000J0_001/R3_A0000J0_001_final.md
전체 길이: 17,974 chars

--- 미리보기 (첫 2000자) ---
<page-001>
<table class="page-meta">
  <tr>
    <td>page_number</td>
    <td>1</td>
  </tr>
  <tr>
    <td>page_summary</td>
    <td></td>
  </tr>
  <tr>
    <td>entities</td>
    <td></td>
  </tr>
</table>


<간이투자설명서>                                                                                    기준일: 2025.02.16 
한화 PLUS 한화그룹주증권상장지수투자신탁(주식) [펀드코드: EG315] 
투자 위험 등급 2등급 [높은 위험] 
한화자산운용(주)는 이 투자신탁의 투자대상자산의 종류 및 위험도 
등을 감안하여 2등급으로 분류하였습니다. 
이 투자신탁은 「예금자보호법」에 의한 보호를 받지 않는 실적배당상품으로, 
1좌당 순자산가치의 일간 변동률을 “FnGuide 한화그룹주 지수”의 1배수와 
유사하도록 운용하는 것을 목적으로 하고, 주식가격 변동위험, 특정 기업집
단(그룹)에 속하는 주식 투자에 따른 특정주식 집중 투자위험, 파생상품 투
자위험, 추적오차 발생위험, 지수산출방식의 대폭변경 또는 중단 위험, 상장
폐지위험 등이 있으므로 투자에 신중을 기하여 주시기 바랍니다. 
1 
2 
3 
4 
5 
6 
매우 
높은 
위험 
높은 
위험 
다소 
높은 
위험 
보통 
위험 
낮은 
위험 
매우 
낮은 
위험 
이 요약정보는 한화 PLUS 한화그룹주증권상장지수투자신탁(주식)의 투자설명서의 내용 중 중요사항을 발췌·요약한 핵심정
보를 담고 있습니다. 따라

---
## 4. 정리

In [9]:
# 임시 파일 정리
import shutil

shutil.rmtree(tmp_dir, ignore_errors=True)
print(f"임시 파일 삭제: {tmp_dir}")

NameError: name 'tmp_dir' is not defined

## 파이프라인 요약

### IBM OCR API 사용법

**`/ocr` — OCR + bbox 시각화만 (LLM 없음)**
```python
requests.post("/ocr", json={
    "inputPath": "s3://bucket/input/sample.pdf",
    "outputPath": "s3://bucket/output/",
    "modelId": "ibm",
    "ibmConfidenceThreshold": 0.3,
    "generateBboxImages": True,
})
```

**`/process` — 전체 파이프라인 (OCR + LLM 요약 + 마크다운)**
```python
requests.post("/process", json={
    "inputPath": "s3://bucket/input/sample.pdf",
    "outputPath": "s3://bucket/output/",
    "llmModelId": "ap-northeast-2.anthropic.claude-haiku-4-5-20251001-v1:0",
    "modelId": "ibm",
    "ibmConfidenceThreshold": 0.3,
    "noSummary": False,
})
```

### Docling vs IBM OCR 비교

| 항목 | Docling | IBM LayoutPredictor |
|---|---|---|
| 레이아웃 감지 | DocLayoutNet | IBM egret-large |
| 텍스트 추출 | Docling 내부 | PyMuPDF (fitz) |
| 테이블 구조 | TableFormer (MD 생성) | bbox 크롭 이미지만 |
| figure 분류 | EfficientNet-B0 (16 클래스) | label 그대로 사용 |
| 레이블 종류 | Table, Picture 중심 | 15종 (Section-header 등 포함) |
| 속도 | 느림 (모델 로딩 포함) | 빠름 |
| figure/table 감지 수 | 적음 | 많음 (confidence 조정 가능) |